# disposal_rules

Executable notebook module. English explanations precede each functional block. Restart the kernel after changing modules.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

__file__ = str(PROJECT / 'disposal_rules.ipynb')


## Imports and configuration

Load required libraries and define the paths, labels and constants used by the following functions.


In [ ]:
"""Source-linked household guidance, separate from visual model predictions."""
VERIFIED = "2026-10-23"  # Record the historical source-check date, not a live verification.
RECYCLE = "https://www.cityofsydney.nsw.gov.au/waste-recycling-services/recycle-household-items"
GENERAL = "https://www.cityofsydney.nsw.gov.au/explainers/2026/moving-in-or-out-waste-and-recycling-guide"
DOORSTEP = "https://www.cityofsydney.nsw.gov.au/environmental-support-funding/doorstep-recycling"
FOOD = "https://www.cityofsydney.nsw.gov.au/waste-recycling-services/prepare-for-your-food-scraps-recycling-collection"
GARDEN = "https://www.cityofsydney.nsw.gov.au/waste-recycling-services/recycle-garden-organics"
EPA = "https://www.epa.nsw.gov.au/Your-environment/Recycling-and-reuse/household-recycling-overview/community-recycling-centres"
EPA_OVERVIEW = "https://www.epa.nsw.gov.au/Your-environment/Recycling-and-reuse/household-recycling-overview"

ITEM_TYPES = ["Not confirmed", "Empty plastic water/drink bottle", "Empty non-black rigid plastic food container", "Empty glass bottle or jar", "Empty aluminium or steel can", "Paper or cardboard with food scraps removed", "Ceramic crockery", "Soft plastic bag or wrapper", "Polystyrene packaging", "Black plastic container", "Food scraps", "Garden leaves or small prunings", "Battery", "Small electronics", "Textiles", "Unwanted medicines", "Other / unsure"]



## predicted_item_guidance

Legacy helper, not called by the current material-only application.


In [ ]:
def predicted_item_guidance(label, object_name, food_service=False, uncertain=False):  # Legacy helper; the current material-only application does not call this function.
    """Route inferred objects conditionally, without claiming user confirmation."""
    if uncertain:
        return disposal_guidance(label, food_service=food_service, uncertain=True)
    specific = {
        "plastic drink bottle": ITEM_TYPES[1], "plastic food container": ITEM_TYPES[2],
        "glass bottle": ITEM_TYPES[3], "glass jar": ITEM_TYPES[3],
        "aluminium beverage can": ITEM_TYPES[4], "steel food tin": ITEM_TYPES[4],
        "metal food can": ITEM_TYPES[4], "empty tin can": ITEM_TYPES[4],
        "plastic bag": "Soft plastic bag or wrapper",
        "black plastic food container": "Black plastic container",
    }
    item = specific.get(object_name, "Not confirmed")
    if label in {"paper", "cardboard"}:
        item = ITEM_TYPES[5]
    if label == "trash" and object_name in {"used tissue", "disposable nappy", "cigarette butt", "disposable face mask", "paper kitchen towel", "roll of paper kitchen towel", "paper napkin"}:
        return "General waste - red lid bin", f"### General waste - red lid bin\n\nConditional on the suggested item being correct. Not yellow lid recycling.\n\n[Official source]({RECYCLE}) | Checked {VERIFIED}"
    route, text = disposal_guidance(label, item, food_service)
    text = text.replace("User-confirmed item guidance", "Conditional guidance for the inferred object")
    if item in ITEM_TYPES[1:6]:
        text += "\n\nConfirm that this is the named item, not black plastic, coated mixed packaging, or contaminated packaging, before using this route."
    return route, text


## disposal_guidance

Apply conditional local rules using material, user confirmation and service availability.


In [ ]:
def disposal_guidance(label, item_type="Not confirmed", food_service=False, uncertain=False):
    if item_type not in ITEM_TYPES:
        raise ValueError("Unknown item confirmation")
    confirmed = item_type != "Not confirmed"  # Distinguish explicit human input from an unconfirmed model suggestion.
    source = RECYCLE
    if item_type in ITEM_TYPES[1:6]:  # Handle the five explicitly confirmed accepted-item types.
        route, advice = "Recyclable - yellow lid bin", "Empty and dry the accepted container, or remove food scraps from paper/cardboard. Leave bottle lids and labels on."
    elif item_type in {"Soft plastic bag or wrapper", "Black plastic container"}:
        route, advice, source = "General waste - red lid bin", "Not accepted in the yellow lid bin. Use the red lid household rubbish bin.", GENERAL
    elif item_type == "Polystyrene packaging":
        route, advice, source = "Special recycling - check collection conditions", "Clean, dry, rigid packaging foam may qualify for doorstep recycling. Food trays and bendy foam do not qualify for that service.", DOORSTEP
    elif item_type == "Battery" or (not confirmed and label == "battery"):  # Handle a confirmed battery or an unconfirmed battery prediction.
        route, advice, source = "Problem waste - battery drop-off", "Never put batteries in household bins. Use an appropriate battery drop-off or CRC; confirm acceptance for large or damaged batteries.", EPA
    elif item_type == "Small electronics" or (not confirmed and label == "ewaste"):
        route, advice, source = "Special recycling - e-waste collection", "Not for the yellow lid bin. Check council collection size limits and delete personal device data before recycling.", DOORSTEP
    elif item_type == "Unwanted medicines" or (not confirmed and label == "medicines"):
        route, advice, source = "Special disposal - community pharmacy", "Take unwanted medicines to a community pharmacy. Confirm packaging and sharps separately; do not infer safe disposal from packaging appearance.", EPA_OVERVIEW
    elif not confirmed and label in {"household_problem_waste", "light_bulbs"}:
        route, advice, source = "Problem waste - identify item and collection point", "Do not put in household recycling. Confirm the exact item and use an accepting CRC, Chemical CleanOut or council special service.", EPA
    elif item_type == "Food scraps" or (not confirmed and label == "food_organics"):
        if food_service:  # Use the user-provided service-access flag.
            route, advice = "Organics - food scraps service", "Use the maroon lid food scraps bin for accepted unpackaged food. No plastic packaging, garden waste or hard shells."
        else:
            route, advice = "Food organics - confirm service; otherwise red lid", "Check access to a household or community food scraps service. Without a suitable service, food goes in general waste, not the garden organics bin."
        source = FOOD
    elif item_type == "Garden leaves or small prunings" or (not confirmed and label == "vegetation"):
        route, advice, source = "Garden organics - green lid bin", "For leaves, grass and accepted small prunings. Not food, plant pots, soil, manufactured wood or large logs.", GARDEN
    elif item_type == "Textiles" or (not confirmed and label == "textile_trash"):
        route, advice, source = "Reuse or special textile recycling", "Donate reusable items. Eligible unusable textiles must be clean and dry for doorstep collection; check exclusions. Not yellow lid recycling.", DOORSTEP
    elif item_type == "Ceramic crockery" or (not confirmed and label == "ceramics"):
        route, advice = "Non-recyclable in yellow lid - confirm disposal", "Crockery and ceramics are not accepted in yellow lid recycling. Consider reuse or check council item-specific disposal; do not treat ceramic as recyclable container glass."
    elif not confirmed and label in {"plastic", "glass", "metal", "paper", "cardboard"}:  # Material labels alone cannot establish an accepted recyclable item.
        route, advice = "Potentially recyclable - confirm item type", "Material alone is insufficient. Yellow lid accepts specified bottles, jars, cans, paper and cardboard, not every object made from these materials. Select the matching confirmed item type."
    elif not confirmed and label == "wood":
        route, advice, source = "Wood item - check council disposal", "Manufactured wooden items are not accepted in the green lid garden organics bin. Check reuse, bulky collection or item-specific disposal.", GARDEN
    else:
        route, advice, source = "Manual review - identify exact item", "Mixed or unidentified objects need item-specific checking. Do not assume general rubbish or recycling when batteries, chemicals or sharps may be present.", RECYCLE
    if uncertain and not confirmed:  # Override an unconfirmed route when the model requests review.
        route = "Manual review - disposal not confirmed"
        advice = "The visual classification is uncertain. Confirm the item before following a disposal route. " + advice
    heading = "User-confirmed item guidance" if confirmed else "Conditional guidance for the model suggestion"
    markdown = f"### {route}\n\n{heading}. {advice}\n\n[Official source]({source}) | City of Sydney / NSW EPA | Checked {VERIFIED}\n\nGuidance is conditional on the item and its contents. The image model cannot verify emptiness, contamination, battery condition or hidden hazards."  # Format route, conditions, source, historical check date and limitations.
    return route, markdown
